# Tier Driver Analysis

## Objective

This notebook provides a supporting analysis of customer characteristics and behavioral drivers across predicted CLV tiers.

Built on top of the final tiered customer output table, this analysis:

* Examines the distribution of customers and value across CLV tiers
* Compares behavioral driver prevalence across tiers
* Identifies key differentiators between high-value and low-value customer segments
* Provides business interpretation to inform targeting and campaign strategy

This notebook does not retrain models or repeat CLV construction logic. It focuses exclusively on understanding what makes higher-value customers different based on the final predicted CLV segmentation.

## 1. Load Tiered Customer Data

Load the final tiered customer output table that contains predicted CLV tiers and behavioral driver flags.

In [0]:
from pyspark.sql import functions as F
import pandas as pd

# Configuration
OUTPUT_SCHEMA = "dev_datalake_insight_analytics_spain"
TIERED_TABLE = f"{OUTPUT_SCHEMA}.clv_forecast_tiered_2024_12_31_v5"

# Driver columns to analyze
DRIVER_COLS = [
    "is_omnichannel",
    "is_contactable",
    "is_ump_member",
    "is_multi_category_9p_universes",
    "is_fast_second_purchase_60d",
    "is_recent_buyer_60d",
    "has_ecosystem_engagement",
    "is_app_user",
    "is_low_returns"
]

print("Configuration loaded")
print(f"Tiered table: {TIERED_TABLE}")
print(f"Driver columns: {len(DRIVER_COLS)}")

Configuration loaded
Tiered table: dev_datalake_insight_analytics_spain.clv_forecast_tiered_2024_12_31_v5
Driver columns: 9


In [0]:
# Load the final tiered customer table
print("Loading final tiered customer table...\n")

df_tiered = spark.table(TIERED_TABLE)

total_customers = df_tiered.count()
print(f"Total customers: {total_customers:,}")

# Show sample
print("\nSample data:")
df_tiered.select(
    "member_id", 
    "tier", 
    "value_decile", 
    "pred_freq_12m", 
    "pred_clv_3y_global"
).show(5, truncate=False)

Loading final tiered customer table...

Total customers: 217,665

Sample data:
+------------------------------------+------+------------+------------------+------------------+
|member_id                           |tier  |value_decile|pred_freq_12m     |pred_clv_3y_global|
+------------------------------------+------+------------+------------------+------------------+
|0055965c-8ec9-412a-b6b4-f094628fc721|Silver|7           |0.8383668745186668|123.04679849246176|
|00abc7e8-47c7-4873-b396-0526bbf00938|Gold  |2           |5.404220080802162 |793.1753979105637 |
|00f363d1-8a43-450f-9307-5e1a17639961|Silver|6           |1.2902274345860187|189.3662070607215 |
|014a3fb3-265b-4664-9a0d-59593f5ca54f|Bronze|9           |0.8055313293240697|118.22753757489585|
|026fbdef-5de6-44b3-bc36-eaed26c2c59d|Silver|6           |1.4881658704190284|218.4175586445264 |
+------------------------------------+------+------------+------------------+------------------+
only showing top 5 rows


## 2. Tier Distribution Overview

Examine the distribution of customers and predicted CLV value across tiers.

In [0]:
# Calculate tier distribution and value metrics
print("Tier Distribution and Value Summary\n")

# Calculate total CLV for share calculation
total_clv = df_tiered.agg(F.sum("pred_clv_3y_global").alias("total")).collect()[0]["total"]

# Tier-level summary
tier_summary = df_tiered.groupBy("tier").agg(
    F.count("*").alias("n_customers"),
    F.avg("pred_freq_12m").alias("avg_predicted_freq"),
    F.avg("pred_clv_3y_global").alias("avg_predicted_clv"),
    F.sum("pred_clv_3y_global").alias("total_clv")
).withColumn(
    "pct_of_total_clv",
    F.round((F.col("total_clv") / total_clv * 100), 1)
).withColumn(
    "pct_of_customers",
    F.round((F.col("n_customers") / total_customers * 100), 1)
).orderBy(F.desc("avg_predicted_clv"))

print("Tier Distribution:")
display(tier_summary)

print(f"\nTotal customers: {total_customers:,}")
print(f"Total predicted 3-year CLV: EUR {total_clv:,.0f}")

Tier Distribution and Value Summary

Tier Distribution:


tier,n_customers,avg_predicted_freq,avg_predicted_clv,total_clv,pct_of_total_clv,pct_of_customers
Platinum,23783,8.893297698133686,1305.2660393148253,3.104314221302449E7,37.6,10.9
Gold,64556,3.3366165606082583,489.7139880621952,3.1613976213343073E7,38.3,29.7
Silver,88752,1.2357951877775284,181.37720617328526,1.6097589802291414E7,19.5,40.8
Bronze,40574,0.639710450447767,93.89006803848989,3809495.620593689,4.6,18.6



Total customers: 217,665
Total predicted 3-year CLV: EUR 82,564,204


## 3. Driver Comparison Across Tiers

Analyze how behavioral driver prevalence varies across CLV tiers.

In [0]:
# Calculate driver prevalence (adoption rate) by tier
print("Driver Prevalence Across Tiers\n")

# Build aggregation for all drivers
driver_agg_exprs = []
for driver in DRIVER_COLS:
    driver_agg_exprs.append(
        F.round((F.sum(F.col(driver).cast("int")) / F.count("*") * 100), 1).alias(driver)
    )

# Calculate prevalence by tier
df_driver_prevalence = df_tiered.groupBy("tier").agg(*driver_agg_exprs)

# Order by tier value (assuming tier order: Platinum > Gold > Silver > Bronze)
tier_order = ["Platinum", "Gold", "Silver", "Bronze"]
df_driver_prevalence = df_driver_prevalence.orderBy(
    F.when(F.col("tier") == "Platinum", 1)
    .when(F.col("tier") == "Gold", 2)
    .when(F.col("tier") == "Silver", 3)
    .when(F.col("tier") == "Bronze", 4)
    .otherwise(5)
)

print("Driver Prevalence by Tier (% of customers with driver):")
display(df_driver_prevalence)

print("\nInterpretation:")
print("- Values represent the percentage of customers in each tier who have the driver enabled (value = 1)")
print("- Higher percentages in upper tiers indicate drivers associated with higher CLV")

Driver Prevalence Across Tiers

Driver Prevalence by Tier (% of customers with driver):


tier,is_omnichannel,is_contactable,is_ump_member,is_multi_category_9p_universes,is_fast_second_purchase_60d,is_recent_buyer_60d,has_ecosystem_engagement,is_app_user,is_low_returns
Platinum,70.8,54.9,2.2,66.0,80.1,84.3,5.4,20.0,56.8
Gold,43.8,55.3,0.9,0.6,56.0,49.6,3.7,7.8,72.6
Silver,16.0,39.4,0.3,0.0,23.8,11.3,1.1,2.7,93.7
Bronze,20.0,46.4,0.5,0.0,28.9,0.4,1.0,3.0,96.1



Interpretation:
- Values represent the percentage of customers in each tier who have the driver enabled (value = 1)
- Higher percentages in upper tiers indicate drivers associated with higher CLV


## 4. High vs Low Value Comparison

Focus on key differentiators between high-value (Platinum, Gold) and low-value (Bronze, Silver) segments.

In [0]:
# Compare high-value (Platinum, Gold) vs low-value (Bronze, Silver) tiers
print("High-Value vs Low-Value Tier Comparison\n")

# Define segments
df_high_value = df_tiered.filter(F.col("tier").isin(["Platinum", "Gold"]))
df_low_value = df_tiered.filter(F.col("tier").isin(["Bronze", "Silver"]))

high_count = df_high_value.count()
low_count = df_low_value.count()

print(f"High-value segment: {high_count:,} customers (Platinum + Gold)")
print(f"Low-value segment: {low_count:,} customers (Bronze + Silver)\n")

# Calculate comparison metrics
comparison_data = []

for driver in DRIVER_COLS:
    high_pct = df_high_value.agg(
        (F.sum(F.col(driver).cast("int")) / F.count("*") * 100).alias("pct")
    ).collect()[0]["pct"]
    
    low_pct = df_low_value.agg(
        (F.sum(F.col(driver).cast("int")) / F.count("*") * 100).alias("pct")
    ).collect()[0]["pct"]
    
    absolute_diff = high_pct - low_pct
    lift = (absolute_diff / low_pct * 100) if low_pct > 0 else 0
    
    comparison_data.append({
        "driver": driver,
        "high_value_pct": round(high_pct, 1),
        "low_value_pct": round(low_pct, 1),
        "absolute_diff_pct": round(absolute_diff, 1),
        "lift_pct": round(lift, 1)
    })

df_comparison = spark.createDataFrame(comparison_data).orderBy(F.desc("lift_pct"))

print("Driver Comparison (High-Value vs Low-Value):")
display(df_comparison)

print("\nInterpretation:")
print("- Positive lift indicates the driver is more prevalent in high-value tiers")
print("- Higher absolute difference suggests stronger differentiation")
print("- Top drivers by lift represent key behaviors associated with higher CLV")

High-Value vs Low-Value Tier Comparison

High-value segment: 88,339 customers (Platinum + Gold)
Low-value segment: 129,326 customers (Bronze + Silver)

Driver Comparison (High-Value vs Low-Value):


absolute_diff_pct,driver,high_value_pct,lift_pct,low_value_pct
18.2,is_multi_category_9p_universes,18.2,471065.4,0.0
51.0,is_recent_buyer_60d,58.9,648.4,7.9
8.3,is_app_user,11.1,297.0,2.8
3.1,has_ecosystem_engagement,4.2,285.1,1.1
0.9,is_ump_member,1.3,274.2,0.3
33.8,is_omnichannel,51.1,195.8,17.3
37.1,is_fast_second_purchase_60d,62.5,145.9,25.4
13.6,is_contactable,55.2,32.7,41.6
-26.1,is_low_returns,68.4,-27.6,94.5



Interpretation:
- Positive lift indicates the driver is more prevalent in high-value tiers
- Higher absolute difference suggests stronger differentiation
- Top drivers by lift represent key behaviors associated with higher CLV


## 5. Business Interpretation

Identify actionable insights for targeting and campaign strategy.

In [0]:
# Identify customers with high CLV potential but missing key drivers
print("Driver Gap Analysis - Targeting Opportunities\n")

# Focus on Gold and Silver tiers (most actionable for campaigns)
df_actionable = df_tiered.filter(F.col("tier").isin(["Gold", "Silver"]))

# Get top 3 drivers by lift from previous analysis
top_drivers_list = df_comparison.orderBy(F.desc("lift_pct")).limit(3).select("driver").rdd.flatMap(lambda x: x).collect()

print(f"Analyzing gaps for top 3 differentiating drivers: {top_drivers_list}\n")

gap_analysis = []

for driver in top_drivers_list:
    for tier in ["Gold", "Silver"]:
        # Count customers WITHOUT the driver
        gap_count = df_actionable.filter(
            (F.col("tier") == tier) & (F.col(driver) == 0)
        ).count()
        
        # Average CLV of customers without the driver
        avg_clv_without = df_actionable.filter(
            (F.col("tier") == tier) & (F.col(driver) == 0)
        ).agg(F.avg("pred_clv_3y_global")).collect()[0][0]
        
        tier_total = df_actionable.filter(F.col("tier") == tier).count()
        gap_pct = (gap_count / tier_total * 100) if tier_total > 0 else 0
        
        gap_analysis.append({
            "tier": tier,
            "driver": driver,
            "customers_without_driver": gap_count,
            "pct_gap": round(gap_pct, 1),
            "avg_clv_eur": round(avg_clv_without, 0) if avg_clv_without else 0
        })

df_gaps = spark.createDataFrame(gap_analysis).orderBy("tier", F.desc("customers_without_driver"))

print("Target Audience Sizes (Customers Missing Key Drivers):")
display(df_gaps)

print("\nActionable Insight:")
print("These represent customers in valuable tiers who lack key behavioral drivers.")
print("Targeted campaigns to activate these behaviors could drive CLV uplift.")
print("Focus on the largest gap sizes for maximum campaign impact.")

Driver Gap Analysis - Targeting Opportunities

Analyzing gaps for top 3 differentiating drivers: ['is_multi_category_9p_universes', 'is_recent_buyer_60d', 'is_app_user']

Target Audience Sizes (Customers Missing Key Drivers):


avg_clv_eur,customers_without_driver,driver,pct_gap,tier
488.0,64164,is_multi_category_9p_universes,99.4,Gold
486.0,59542,is_app_user,92.2,Gold
458.0,32564,is_recent_buyer_60d,50.4,Gold
181.0,88747,is_multi_category_9p_universes,100.0,Silver
181.0,86372,is_app_user,97.3,Silver
173.0,78746,is_recent_buyer_60d,88.7,Silver



Actionable Insight:
These represent customers in valuable tiers who lack key behavioral drivers.
Targeted campaigns to activate these behaviors could drive CLV uplift.
Focus on the largest gap sizes for maximum campaign impact.


## 6. Conclusion

This analysis reveals clear behavioral patterns that differentiate high-value from low-value customers:

**Key Differentiators:**
* Higher-tier customers show significantly greater adoption of omnichannel behaviors, mobile app usage, and ecosystem engagement
* Fast second purchase behavior and multi-category shopping are strong indicators of higher lifetime value
* Contactability and UMP membership also show positive correlation with tier level

**Business Implications:**
* **Targeting Opportunities**: Focus campaigns on moving customers up the driver adoption curve, particularly in mobile app activation and omnichannel engagement
* **Segment Prioritization**: High-value customers with driver gaps represent the highest ROI opportunities for intervention
* **Risk Mitigation**: Customers in higher tiers with declining driver scores should be monitored for churn risk
* **Campaign Design**: Driver-specific campaigns can be designed to close identified gaps and move customers toward higher-value behaviors

This tier driver analysis provides a foundation for actionable segmentation strategies that go beyond historical purchase patterns to focus on forward-looking behavioral engagement.